# OAC Catalog ACL Extractor — Bronze Layer

**Version:** 2.0 — two-stage write (Standard Catalog + ADW)  
**Purpose:** Pull ACLs for all catalog item types from Oracle Analytics Cloud via REST API, stage to OCI Object Storage via the AIDP Standard Catalog (Delta), then write to ADW via the AIDP External Catalog.

**Stage 1 target:** `cbtest_standard_catalog.default.OAC_CATALOG_ACL_BRONZE` (Delta)  
**Stage 2 target:** `arganoadw_oacuser_sh.oacuser.OAC_CATALOG_ACL` (ADW)  
**Auth:** `tokens.json` downloaded from OAC Profile — no `client_secret` or IDCS configuration required

---
### Pre-requisites
1. OAC: click name badge → Profile → Access Tokens → Download tokens → upload `tokens.json` to `/Workspace/Shared/`
   - If Access Tokens tab not visible: Profile → Advanced → Enable Developer Options → Save
2. Calling user must have OAC **BI Service Administrator** role
3. `cbtest_standard_catalog` registered in AIDP with write access to the backing OCI Object Storage bucket
4. `arganoadw_oacuser_sh` External Catalog registered in AIDP Master Catalog
5. `oacuser` schema pre-exists in ADW — do **not** attempt `CREATE SCHEMA`
6. Spark cluster attached to this notebook

---
### Architecture Note
Stage 2 reads from the Delta table written in Stage 1 rather than the in-memory DataFrame. This validates the object storage write before touching ADW. If Stage 1 fails, Stage 2 will not run — preventing a partial or stale ADW load.

## Section 1 — Imports & Configuration
All configuration is centralised here so the notebook can be pointed at a different OAC instance, Standard Catalog, or ADW target without touching logic in Sections 2–8.

- **`TOKENS_FILE`** — path to `tokens.json` in the AIDP workspace. The `OACTokenManager` in Section 2 handles auto-refresh so long-running extracts do not fail mid-run.
- **`CLIENT_ID`** — OAC instance's built-in IDCS app ID from the JWT. The `_APPID` suffix is stripped automatically before calling the refresh endpoint.
- **`STANDARD_CATALOG_TABLE`** — Stage 1 Delta target. The Standard Catalog is backed by OCI Object Storage and supports Delta format. Table is created automatically on first run if it doesn't exist.
- **`ADW_FULL_PATH`** — Stage 2 ADW target via External Catalog. The `oacuser` schema must pre-exist in ADW. See Section 7 for External Catalog write constraints.
- **`CATALOG_TYPES`** — the five types cover the full set of ACL-managed assets in OAC.
- **`PAGE_SIZE`** — 100 is the practical maximum per page. Total pages read from `oa-page-count` response header.
- **`RATE_LIMIT_WAIT`** — 0.2s sleep between ACL calls. Increase if the OAC instance throttles.
- **`TOKEN_REFRESH_BUFFER`** — tokens refreshed proactively 300s before expiry.

In [ ]:
import requests
import pandas as pd
import base64
import time
from datetime import datetime, timezone

# ── OAC Connection ───────────────────────────────────────────
OAC_BASE_URL    = 'https://argano4oracleanalytics-idsmdul6idrs-ia.analytics.ocp.oraclecloud.com'
OAC_API_VERSION = '20210901'

# ── Token Configuration ──────────────────────────────────────
# How to get tokens.json:
#   OAC Home → click your name badge (top right)
#   → Profile → Access Tokens → Download tokens
#   Upload to /Workspace/Shared/ and set path below.
TOKENS_FILE     = '/Workspace/Shared/tokens.json'
IDCS_DOMAIN_URL = 'https://idcs-55a83f44a5c945af86ee0605a1856068.identity.oraclecloud.com'
CLIENT_ID       = 'gkligdfeuzql4yw7pb74ka6ecx3rjsga_APPID'

# ── Stage 1: Standard Catalog (Delta / Object Storage) ───────
# Delta format — supports schema evolution, ACID, time travel.
# Table created automatically on first run if it doesn't exist.
STANDARD_CATALOG_TABLE = 'cbtest_standard_catalog.default.OAC_CATALOG_ACL_BRONZE'

# ── Stage 2: ADW External Catalog ────────────────────────────
# oacuser schema must pre-exist in ADW. Do NOT CREATE SCHEMA.
ADW_CATALOG   = 'arganoadw_oacuser_sh'
ADW_SCHEMA    = 'oacuser'
ADW_TABLE     = 'OAC_CATALOG_ACL'
ADW_FULL_PATH = f'{ADW_CATALOG}.{ADW_SCHEMA}.{ADW_TABLE}'

# ── Catalog Types to Extract ─────────────────────────────────
CATALOG_TYPES = ['workbooks', 'folders', 'datasets', 'dataflows', 'connections']

# ── Pagination & Rate Limiting ────────────────────────────────
PAGE_SIZE            = 100
RATE_LIMIT_WAIT      = 0.2
TOKEN_REFRESH_BUFFER = 300

print('=' * 55)
print('  SECTION 1 COMPLETE: Imports & Configuration')
print(f'  OAC    : {OAC_BASE_URL}')
print(f'  Stage 1: {STANDARD_CATALOG_TABLE}')
print(f'  Stage 2: {ADW_FULL_PATH}')
print(f'  Types  : {CATALOG_TYPES}')
print('=' * 55)

## Section 2 — Token Manager
Manages the full OAuth 2.0 token lifecycle for OAC API calls.

**Why `tokens.json` instead of client credentials?** The OAC Catalog/ACL APIs require a user-context Bearer token — client credentials alone are not sufficient. Downloading `tokens.json` from OAC Profile requires no IDCS app configuration.

**Critical discovery — refresh endpoint:** The correct endpoint is on the **OAC hostname**, not IDCS. Calling IDCS `/oauth2/v1/token` for refresh returns 401 Unauthorized.
```
POST /api/dv/api/v1/tokens/token/refresh
Authorization: Bearer <current_access_token>
Content-Type:  text/plain
Body:          <refresh_token>  (plain text, not JSON)
```
The `_APPID` suffix is stripped from `CLIENT_ID` before calling — the endpoint expects the raw GUID only.

**Auto-refresh:** Every call to `token_mgr.token` checks time remaining. If within `TOKEN_REFRESH_BUFFER` seconds of expiry, `_refresh()` is called proactively before returning the token.

In [ ]:
class OACTokenManager:

    def __init__(self, tokens_file):
        self._access_token  = None
        self._refresh_token = None
        self._expires_at    = 0
        self._load_tokens(tokens_file)

    def _load_tokens(self, tokens_file):
        import json as _json
        with open(tokens_file, 'r') as f:
            data = _json.load(f)
        self._access_token  = data.get('access_token')  or data.get('accessToken')
        self._refresh_token = data.get('refresh_token') or data.get('refreshToken')
        if not self._access_token:
            raise ValueError('❌ tokens.json must contain access_token. Re-download from OAC Profile.')
        if not self._refresh_token:
            raise ValueError('❌ tokens.json must contain refresh_token. Download (not copy) from OAC Profile.')
        try:
            payload = self._access_token.split('.')[1]
            payload += '=' * (4 - len(payload) % 4)
            claims = _json.loads(base64.urlsafe_b64decode(payload))
            self._expires_at = claims.get('exp', 0)
            expiry_str = datetime.fromtimestamp(self._expires_at, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
            remaining = self.seconds_remaining
            status = '✅ Valid' if remaining > TOKEN_REFRESH_BUFFER else '⚠️  Expired or expiring soon'
            print(f'  Token status   : {status}')
            print(f'  Expires at     : {expiry_str}')
            print(f'  Time remaining : {remaining}s')
        except Exception:
            self._expires_at = time.time() + 3600
            print('  Token expiry   : unknown (will refresh proactively)')

    def _refresh(self):
        refresh_url = f'{OAC_BASE_URL}/api/dv/api/v1/tokens/token/refresh'
        resp = requests.post(
            refresh_url,
            data=self._refresh_token,
            headers={'Authorization': f'Bearer {self._access_token}', 'Content-Type': 'text/plain'},
            timeout=30
        )
        if resp.status_code == 401:
            raise RuntimeError(
                '❌ Refresh token rejected (401).\n'
                '   tokens.json has expired. Re-download from OAC Profile and re-upload.'
            )
        resp.raise_for_status()
        data = resp.json()
        self._access_token = data.get('accessToken') or data.get('access_token')
        if not self._access_token:
            raise ValueError(f'❌ Refresh response missing accessToken. Response: {data}')
        if   'refreshToken'  in data: self._refresh_token = data['refreshToken']
        elif 'refresh_token' in data: self._refresh_token = data['refresh_token']
        expires_in       = int(data.get('expiresIn') or data.get('expires_in') or 3600)
        self._expires_at = time.time() + expires_in
        expiry_str = datetime.fromtimestamp(self._expires_at, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
        print(f'🔑 Token refreshed. Valid for {expires_in}s → expires at {expiry_str}')

    @property
    def token(self):
        time_remaining = self._expires_at - time.time()
        if time_remaining <= TOKEN_REFRESH_BUFFER:
            print(f'⚠️  Token expiring in {int(time_remaining)}s — refreshing...')
            self._refresh()
        return self._access_token

    def get_headers(self):
        return {'Authorization': f'Bearer {self.token}', 'Content-Type': 'application/json'}

    @property
    def seconds_remaining(self):
        return max(0, int(self._expires_at - time.time()))


print('Loading tokens.json...')
token_mgr = OACTokenManager(TOKENS_FILE)
print('=' * 55)
print('  SECTION 2 COMPLETE: Token Manager Ready')
print('=' * 55)

## Section 3 — OAC API Helpers
Three utility functions used by the extraction loop. All API calls go through these helpers so token refresh, error handling, and encoding logic are defined once and reused consistently.

- **`get_headers()`** — returns Authorization and Content-Type headers. Calls `token_mgr.token` which triggers proactive refresh if near expiry.
- **`b64_encode_id()`** — Base64 URL-safe encodes catalog object paths for the `getACL` endpoint. Trailing `=` padding stripped as it's not valid in URL segments. Mirror of Silver's `decode_base64_id` UDF.
- **`get_catalog_items()`** — paginates through all items of a catalog type using `search=*`. Reads `oa-page-count` response header for total pages. Logs token remaining time per page for monitoring during large extracts.
- **`get_acl_for_item()`** — calls the `getACL` action endpoint for a single item. `403` and `404` handled silently — returns empty list so extraction continues without interruption.

In [ ]:
def get_headers(token=None):
    if token:
        return {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
    return token_mgr.get_headers()


def b64_encode_id(object_path):
    # URL-safe Base64 encode. Padding = stripped for URL safety.
    return base64.urlsafe_b64encode(
        object_path.encode('utf-8')
    ).decode('utf-8').rstrip('=')


def get_catalog_items(catalog_type):
    items = []
    page  = 1
    base  = f'{OAC_BASE_URL}/api/{OAC_API_VERSION}/catalog/{catalog_type}'
    while True:
        params = {'search': '*', 'limit': PAGE_SIZE, 'page': page}
        resp   = requests.get(base, headers=get_headers(), params=params, timeout=30)
        if resp.status_code == 404:
            print(f'  ⚠️  {catalog_type}: no items found (404) — skipping')
            break
        resp.raise_for_status()
        page_items = resp.json()
        if not page_items:
            break
        items.extend(page_items)
        total_pages = int(resp.headers.get('oa-page-count', 1))
        print(f'  📄 {catalog_type}: page {page}/{total_pages} — {len(page_items)} items [token: {token_mgr.seconds_remaining}s remaining]')
        if page >= total_pages:
            break
        page += 1
        time.sleep(RATE_LIMIT_WAIT)
    print(f'  ✅ {catalog_type}: {len(items)} total items')
    return items


def get_acl_for_item(catalog_type, item_id):
    # Content-Length: 0 required by OAC API even with empty body.
    url  = f'{OAC_BASE_URL}/api/{OAC_API_VERSION}/catalog/{catalog_type}/{item_id}/actions/getACL'
    hdrs = get_headers()
    hdrs['Content-Length'] = '0'
    resp = requests.post(url, headers=hdrs, timeout=30)
    if resp.status_code in (403, 404):
        return []
    resp.raise_for_status()
    return resp.json()


print('=' * 55)
print('  SECTION 3 COMPLETE: API Helpers Loaded')
print('  Functions: get_headers, b64_encode_id,')
print('             get_catalog_items, get_acl_for_item')
print('=' * 55)

## Section 4 — Extract
Core extraction loop. For each catalog type, all items are fetched first (paginated), then `getACL` is called per item. Results are flattened into **one row per item+principal combination**.

**Field name variations by catalog type:**
- `id` vs `objectId` (connections use `objectId`)
- `path` vs `objectPath`
- `owner` may be a dict with `displayName` or a plain string
- `lastModified` vs `modified`

All handled defensively with `.get()` chains.

**Items with no ACL entries** are still recorded with `NULL` permission fields — preserving catalog coverage. These rows are flagged `MISSING_PERMISSIONS` by the Silver `DATA_QUALITY_FLAG` transformation.

`EXTRACTED_AT` is stamped once at the start of the function and applied to every row for a consistent snapshot timestamp.

In [ ]:
def extract_all_acls():
    all_records  = []
    extracted_at = datetime.now(tz=timezone.utc).isoformat()

    for catalog_type in CATALOG_TYPES:
        print(f'\n{"─" * 55}')
        print(f'🔍 Processing: {catalog_type.upper()}')
        print(f'{"─" * 55}')

        items = get_catalog_items(catalog_type)

        for item in items:
            raw_id   = item.get('id') or item.get('objectId') or ''
            name     = item.get('name', '')
            path     = item.get('path', '') or item.get('objectPath', '')
            owner    = item.get('owner', {})
            owner_nm = owner.get('displayName', '') if isinstance(owner, dict) else str(owner)
            created  = item.get('created', '')
            modified = item.get('lastModified', '') or item.get('modified', '')

            encoded_id = b64_encode_id(raw_id) if raw_id else ''
            if not encoded_id:
                continue

            acl_entries = get_acl_for_item(catalog_type, encoded_id)
            time.sleep(RATE_LIMIT_WAIT)

            base_row = {
                'CATALOG_TYPE': catalog_type, 'ITEM_ID': raw_id,
                'ITEM_NAME': name, 'ITEM_PATH': path,
                'ITEM_OWNER': owner_nm, 'ITEM_CREATED': created,
                'ITEM_MODIFIED': modified, 'EXTRACTED_AT': extracted_at
            }

            if not acl_entries:
                all_records.append({
                    **base_row,
                    'ACCOUNT_GUID': None, 'ACCOUNT_TYPE': None, 'ACCOUNT_NAME': None,
                    'PERM_READ': None, 'PERM_WRITE': None, 'PERM_LIST': None,
                    'PERM_DELETE': None, 'PERM_CHANGE_PERM': None, 'PERM_TAKE_OWN': None
                })
            else:
                for entry in acl_entries:
                    perms = entry.get('permissions', {})
                    all_records.append({
                        **base_row,
                        'ACCOUNT_GUID':     entry.get('accountGuid'),
                        'ACCOUNT_TYPE':     entry.get('accountType'),
                        'ACCOUNT_NAME':     entry.get('accountDisplayName'),
                        'PERM_READ':        int(perms.get('read',             False)),
                        'PERM_WRITE':       int(perms.get('write',            False)),
                        'PERM_LIST':        int(perms.get('list',             False)),
                        'PERM_DELETE':      int(perms.get('delete',           False)),
                        'PERM_CHANGE_PERM': int(perms.get('changePermission', False)),
                        'PERM_TAKE_OWN':    int(perms.get('takeOwnership',    False))
                    })

    print('=' * 55)
    print('  SECTION 4 COMPLETE: Extraction Done')
    print(f'  Total records: {len(all_records):,}')
    print('=' * 55)
    return all_records


print('=' * 55)
print('  SECTION 4 READY: Extract Function Loaded')
print('=' * 55)

## Section 5 — Build Spark DataFrame
Converts the extracted record list into a typed Spark DataFrame with an explicit schema.

**Why explicit schema rather than inference?**
- Column types are deterministic across runs even when null-only columns are present (e.g. `ITEM_CREATED` is blank on this OAC instance — inference could produce `NullType`, breaking the Silver read)
- Integer permission flags (`PERM_*`) must be `IntegerType` for the weighted arithmetic in the Silver `RISK_SCORE` UDF
- Schema mismatches between Stage 1 and Stage 2 are caught here rather than silently at write time

No data preview is shown — printing a DataFrame materialises all rows on the driver, causing memory pressure and increased compute costs at production scale. The quality review gate is Section 5 of the Silver notebook.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.appName('oac_acl_bronze').getOrCreate()

BRONZE_SCHEMA = StructType([
    StructField('CATALOG_TYPE',     StringType(),  True),
    StructField('ITEM_ID',          StringType(),  True),
    StructField('ITEM_NAME',        StringType(),  True),
    StructField('ITEM_PATH',        StringType(),  True),
    StructField('ITEM_OWNER',       StringType(),  True),
    StructField('ITEM_CREATED',     StringType(),  True),
    StructField('ITEM_MODIFIED',    StringType(),  True),
    StructField('ACCOUNT_GUID',     StringType(),  True),
    StructField('ACCOUNT_TYPE',     StringType(),  True),
    StructField('ACCOUNT_NAME',     StringType(),  True),
    StructField('PERM_READ',        IntegerType(), True),
    StructField('PERM_WRITE',       IntegerType(), True),
    StructField('PERM_LIST',        IntegerType(), True),
    StructField('PERM_DELETE',      IntegerType(), True),
    StructField('PERM_CHANGE_PERM', IntegerType(), True),
    StructField('PERM_TAKE_OWN',    IntegerType(), True),
    StructField('EXTRACTED_AT',     StringType(),  True),
])

print('=' * 55)
print('  SECTION 5 READY: Spark Session & Schema Defined')
print(f'  Columns: {len(BRONZE_SCHEMA.fields)}')
print('=' * 55)

## Section 6 — Write Stage 1 (Standard Catalog / Object Storage)
Writes the Bronze DataFrame to the AIDP Standard Catalog as a managed Delta table backed by OCI Object Storage.

**Why Delta for the Standard Catalog?**
- Standard Catalog supports Delta natively
- Provides schema evolution, ACID transactions, and time travel
- Consistent with Silver and Gold layers — full pipeline uses Delta
- Portable: any AIDP instance with a Standard Catalog uses the same write code

**`overwriteSchema=true`** — allows column additions between runs without `DROP TABLE`. Safe because Bronze is always fully regenerated from the API.

> The table is created automatically on first run if it doesn't exist.

In [ ]:
def write_stage1(df):
    print(f'\n[WRITE] Stage 1 — Standard Catalog (Delta / Object Storage)')
    print(f'        Target: {STANDARD_CATALOG_TABLE}')

    (df.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(STANDARD_CATALOG_TABLE)
    )

    df_check  = spark.table(STANDARD_CATALOG_TABLE)
    row_count = df_check.count()
    print('=' * 55)
    print('  SECTION 6 COMPLETE: Stage 1 Write Done')
    print(f'  Table   : {STANDARD_CATALOG_TABLE}')
    print(f'  Rows    : {row_count:,}')
    print(f'  Columns : {len(df_check.columns)}')
    print(f'  Format  : Delta')
    print('=' * 55)
    return row_count


print('=' * 55)
print('  SECTION 6 READY: Stage 1 Write Function Loaded')
print('=' * 55)

## Section 7 — Write Stage 2 (ADW via External Catalog)
Reads from the Stage 1 Delta table and writes to ADW via the AIDP External Catalog. No wallet, no cx_Oracle, no JDBC — Spark resolves the 3-part catalog path through the External Catalog metadata layer.

**Why read from the Delta table rather than the in-memory DataFrame?**  
Validates Stage 1 before touching ADW. If the object storage write failed silently, reading from the Delta table surfaces the problem before ADW is written.

**External Catalog constraints — do NOT violate:**
- **No Delta format** — ADW External Catalogs do not support Delta. `saveAsTable()` without `.format('delta')` uses the correct default.
- **No `CREATE SCHEMA`** — returns 502 Bad Gateway from AIDP Metastore. The `oacuser` schema must pre-exist in ADW.
- **3-part path required** — `catalog.schema.table` all three segments must be present.

In [ ]:
def write_stage2():
    print(f'\n[WRITE] Stage 2 — ADW via External Catalog')
    print(f'        Source: {STANDARD_CATALOG_TABLE}  (Delta)')
    print(f'        Target: {ADW_FULL_PATH}')

    # Read from Stage 1 Delta — validates Stage 1 before ADW write
    df_from_delta = spark.table(STANDARD_CATALOG_TABLE)

    (df_from_delta.write
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(ADW_FULL_PATH)
    )

    df_check = spark.table(ADW_FULL_PATH)
    print('=' * 55)
    print('  SECTION 7 COMPLETE: Stage 2 Write Done')
    print(f'  Table   : {ADW_FULL_PATH}')
    print(f'  Rows    : {df_check.count():,}')
    print(f'  Columns : {len(df_check.columns)}')
    print(f'  Status  : Queryable in AIDP Master Catalog + OAC')
    print('=' * 55)


print('=' * 55)
print('  SECTION 7 READY: Stage 2 Write Function Loaded')
print('=' * 55)

## Section 8 — Run Pipeline
Orchestrates all steps in sequence:

| Step | Action |
|------|--------|
| 1/4 | **Auth** — validate token, print remaining lifetime. If `Expired or expiring soon`, re-download `tokens.json` before proceeding. |
| 2/4 | **Extract** — loop all catalog types, paginate items, call `getACL` per item. Count 0 or unexpectedly low → check token validity and BI Service Administrator role. |
| 3/4 | **Build DataFrame** — typed Spark DataFrame with explicit `BRONZE_SCHEMA`. Schema + partition count logged only — no data preview. |
| 4/4 | **Write** — Stage 1 (Standard Catalog Delta), then Stage 2 (ADW from Delta). Row count + column count confirmed after each stage. |

> **Before running:** ensure `tokens.json` is uploaded to `/Workspace/Shared/` and the Spark cluster is attached.

In [ ]:
print('=' * 60)
print('  OAC CATALOG ACL EXTRACTOR — BRONZE LAYER v2.0')
print(f'  Run time: {datetime.now(tz=timezone.utc).isoformat()}')
print('=' * 60)

# Step 1: Validate token
print('\n[1/4] Authenticating with OAC...')
_ = token_mgr.token
print(f'  ✅ Token valid. {token_mgr.seconds_remaining}s remaining.')

# Step 2: Extract
print('\n[2/4] Extracting catalog items and ACLs...')
records = extract_all_acls()

if not records:
    print('⚠️  No records extracted.')
    print('    Check: token is valid, user has BI Service Administrator role.')
else:
    # Step 3: Build DataFrame
    print(f'\n[3/4] Building Spark DataFrame ({len(records):,} records)...')
    df_bronze = spark.createDataFrame(records, schema=BRONZE_SCHEMA)
    df_bronze.printSchema()
    print(f'  Partitions: {df_bronze.rdd.getNumPartitions()}')

    # Step 4: Write — Stage 1 then Stage 2
    print('\n[4/4] Writing — two-stage pipeline...')
    write_stage1(df_bronze)
    write_stage2()

print('\n' + '=' * 60)
print('  🏁 BRONZE PIPELINE COMPLETE')
print('=' * 60)